# Can we trust AI outputs?

### Learning what deterministic verifiers can, and cannot, guarantee

You've asked an AI model like ChatGPT or whichever large language model (LLM) you reach for, to write a summary, fill in a form, redact a document, and had that small nagging feeling afterward: *is this actually right, or does it just look right?* This notebook walks through one way to answer that question, and shows where that answer stops holding up.

*No coding background required: you'll read, predict, and press run. Every "model attempt" here is a fixed example of what an LLM might output. Nothing calls a live API, so there's no key, account, or cost. The examples were pre-written by Gemini 3.7 Flash.*

## Verifier: something a computer can check

An [IBAN](https://www.iso.org/standard/81090.html) is the account number format banks use to route international payments: two letters for the country, two check digits, then the domestic account number. If you ask a model to invent one, it might hand back `DE44 5001 0517 5407 3249 31`. You can't tell by looking whether it follows the format or is simply invented. You need to check it against a [defined rule](https://www.iso.org/standard/31531.html):

1. Move the first four characters (country code + check digits) to the end of the string
2. Convert every letter to a number (A=10, B=11, ... Z=35)
3. Take the resulting number mod 97 (the remainder after dividing by 97)
4. Remainder of 1 means the IBAN is valid; any other remainder means it is not

For `DE44 5001 0517 5407 3249 31`: rearranged, it reads `500105175407324931` followed by `DE44`. With `D` as 13 and `E` as 14, the whole thing becomes `500105175407324931131444`. Divided by 97, the remainder is 1.

**Quick orientation since this is the notebook's first code cell.** Click the ▶ Run button on this cell to run it. That's all you need to do. Cells rely on the ones above them having already run, so go top to bottom.

The next cell has two parts. The `def` block defines a function, Python's term for a named, reusable recipe, called `passes_iban_check`. Defining it doesn't run anything yet. Python just remembers the steps for later. Given an IBAN as a string (a piece of text), it does exactly what was described above: rearranges it, converts the letters to numbers, takes the result mod 97, and returns `True` or `False`. The `for` loop underneath actually calls the function once for each IBAN and prints the result. Run the cell below to see how two candidate IBANs, `DE44 5001 0517 5407 3249 31` and `DE44 5001 0517 5407 3249 30`, are checked for validity.

In [ ]:
def passes_iban_check(iban: str) -> bool:
    iban = iban.replace(" ", "")

    # move the country code and check digits to the end
    rearranged = iban[4:] + iban[:4]

    # convert letters to numbers - A=10, B=11, ... Z=35
    numeric = "".join(str(int(c, 36)) for c in rearranged)

    # take the whole thing mod 97 - a valid IBAN always leaves remainder 1
    return int(numeric) % 97 == 1


for iban in ["DE44 5001 0517 5407 3249 31", "DE44 5001 0517 5407 3249 30"]:
    print(iban, "->", passes_iban_check(iban))

For this specific case, no model is needed. A dozen lines of arithmetic can answer it exactly, every time.

**That's a verifier:** code that inspects an output and returns a yes/no verdict with a reason.

The rest of this notebook explores what happens when you build a verifier for something messier than a checksum, put it in front of a model's output, and test what that gate can and can't guarantee. You'll run it across four stages, each testing it in a different way, to see what that reveals about verifier coverage and reward hacking. Six short sections carry that arc: closing the loop, watching the fix get fooled and patched twice, plus an optional challenge where you design the coverage yourself.

<div style="text-align:center; margin: 22px 0;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 680 250" style="max-width:660px; width:100%; font-family: sans-serif;">
<line x1="12" y1="200" x2="668" y2="200" stroke="#d9dce0" stroke-width="1"></line>
<rect x="20" y="156" width="116" height="44" rx="6" fill="#909090AB" stroke="#909090AB" stroke-width="0"></rect>
<text x="78" y="183" text-anchor="middle" font-size="12" font-weight="600" fill="#ffffff">REC-001</text>
<text x="78" y="218" text-anchor="middle" font-size="10.5" fill="#5b5f66">a name survives</text>
<text x="78" y="232" text-anchor="middle" font-size="10.5" fill="#1b5e20">caught</text>
<rect x="148" y="142" width="116" height="58" rx="6" fill="#909090AB" stroke="#909090AB" stroke-width="0"></rect>
<text x="206" y="176" text-anchor="middle" font-size="12" font-weight="600" fill="#ffffff">REC-002</text>
<text x="206" y="218" text-anchor="middle" font-size="10.5" fill="#5b5f66">a fact goes missing</text>
<text x="206" y="232" text-anchor="middle" font-size="10.5" fill="#1b5e20">caught</text>
<rect x="276" y="122" width="116" height="78" rx="6" fill="#9F5BD0" stroke="#9F5BD0" stroke-width="1.5"></rect>
<text x="334" y="166" text-anchor="middle" font-size="12" font-weight="600" fill="#ffffff">REC-003</text>
<text x="334" y="218" text-anchor="middle" font-size="10.5" fill="#5b5f66">written as T.L.</text>
<text x="334" y="232" text-anchor="middle" font-size="10.5" fill="#8a6100">missed, patchable</text>
<rect x="404" y="108" width="116" height="92" rx="6" fill="#9F5BD0" stroke="#9F5BD0" stroke-width="1.5"></rect>
<text x="462" y="159" text-anchor="middle" font-size="12" font-weight="600" fill="#ffffff">REC-004</text>
<text x="462" y="218" text-anchor="middle" font-size="10.5" fill="#5b5f66">18,400 vs 18400</text>
<text x="462" y="232" text-anchor="middle" font-size="10.5" fill="#8a6100">false alarm</text>
<rect x="532" y="84" width="116" height="116" rx="6" fill="#000000" stroke="#000000" stroke-width="1.5"></rect>
<text x="590" y="147" text-anchor="middle" font-size="12" font-weight="600" fill="#ffffff">REC-003</text>
<text x="590" y="218" text-anchor="middle" font-size="10.5" fill="#5b5f66">"the only patient…"</text>
<text x="590" y="232" text-anchor="middle" font-size="10.5" fill="#b71c1c">missed, no patch</text>
</svg>
</div>

**Figure 1.** Overview of this notebook's four records across five moments: two caught by the verifier, one missed but correctable, one falsely flagged, and one beyond what string matching can detect.

By the end of this notebook, you'll be able to:

1. Explain why LLM self-grading is unreliable, and how a deterministic check differs.
2. Design a decidable verifier that returns an exact pass/fail on a specific claim.
3. Use that verifier in a generate, verify, retry loop (rejection sampling / self-correction).
4. Recognize the coverage limits of rule-based verification over free text, and why they never close.
5. Explain reward hacking and Goodhart's law: how an imperfect verifier becomes a training reward (RLVR).
6. Generalize the two-layer verification pattern (schema check, then value check) to a task.

A quick term to keep: a claim is [**decidable**](https://doi.org/10.1112/plms/s2-42.1.230) when you can compute the answer exactly from information you already have, with code, no judgment call needed.

## 1. The task

A medical research team wants to share a batch of patient records with a partner lab for a study. Before anything goes out, each record has to lose its direct **identifiers** (patient names, medical record numbers/MRN, exact addresses) while keeping every **clinical fact**, since those facts are the reason to share the data. The plan is to have an AI model rewrite each record: feed it the raw record and get back a [de-identified version](https://doi.org/10.1186/1471-2288-10-70). From here on, you'll check the model's work using a verifier you'll build yourself.

The next cell has three parts. First, `class Record(NamedTuple)` defines the shape every patient record takes: an ID, the raw text, the strings that must disappear, and the facts that must survive. Then `RECORDS` holds the four records you'll work with throughout the notebook, each with its answer key already filled in. `RECORDED_LLM_ATTEMPTS` holds what the model actually wrote when asked to de-identify each record. The last three lines print the first record so you can see its structure before working with it. Run the cell below.

In [ ]:
import textwrap
from typing import List, NamedTuple, Optional

class Record(NamedTuple):
    # One patient record, with its ground truth attached.
    # The ground truth is what makes deterministic verification possible here:
    # we are not asking a model to guess which words were identifying, the
    # clinical team that filed the record already knows.
    record_id: str             # which record this is, e.g. "REC-001"
    text: str                  # the record's full text
    identifiers: List[str]     # must NOT survive de-identification
    clinical_facts: List[str]  # must survive de-identification

RECORDS = {
    "REC-001": Record(
        "REC-001",  # record_id
        "On 14 March 2024, patient Mariana Costa (MRN 88431) was treated for "  # text
        "anaphylaxis lasting 47 minutes, receiving 1204 mL of IV fluids. "
        "Costa was discharged from the emergency ward at 09:12.",
        ["Mariana Costa", "Costa", "88431"],  # identifiers
        ["14 March 2024", "47 minutes", "1204 mL", "09:12"],  # clinical_facts
    ),
    "REC-002": Record("REC-002",
        "Patient record MRN-5510 belongs to Priya Parkinson, admitted at "
        "22:40 after experiencing seizures 3 times in a row. Parkinson was "
        "stabilized within 18 minutes and monitored for 96 hours.",
        ["Priya Parkinson", "Parkinson", "MRN-5510"],
        ["22:40", "3 times", "18 minutes", "96 hours"]),
    "REC-003": Record("REC-003",
        "During the 12 June 2024 admission, patient Tomas Lindqvist "
        "(MRN 21007) experienced a vasovagal episode lasting 210 seconds, "
        "monitored across 4 vital-sign channels. Symptoms cleared without "
        "intervention after 35 minutes.",
        ["Tomas Lindqvist", "Lindqvist", "21007"],
        ["12 June 2024", "210 seconds", "4 vital-sign channels", "35 minutes"]),
    "REC-004": Record("REC-004",
        "At 03:05, patient Yuki Tanaka's monitor triggered an infusion "
        "alarm. A nurse acknowledged the alert within 2 minutes. A "
        "miscalibrated pump had delivered 18400 mcg of medication over 26 "
        "minutes. Tanaka's patient ID, 18402, appears in the medication "
        "administration record next to each dose.",
        ["Yuki Tanaka", "Tanaka", "18402"],
        ["03:05", "2 minutes", "18400 mcg", "26 minutes"]),
}

# Recorded attempts: what the model actually produced when asked to
# de-identify each record. Fixed recordings, not a live model: every
# exercise below runs offline, at no cost. Quoted lines with no comma
# between them are one attempt split across lines; a comma starts the next attempt.
RECORDED_LLM_ATTEMPTS = {
    "REC-001": [
        "On 14 March 2024, patient Costa (MRN 88431) was treated for "
        "anaphylaxis lasting 47 minutes, receiving 1204 mL of IV fluids. "
        "Costa was discharged from the emergency ward at 09:12.",
        "On 14 March 2024, a patient (MRN 88431) was treated for anaphylaxis "
        "lasting 47 minutes, receiving 1204 mL of IV fluids. The patient was "
        "discharged from the emergency ward at 09:12.",
        "On 14 March 2024, a patient was treated for anaphylaxis lasting 47 "
        "minutes, receiving 1204 mL of IV fluids. The patient was discharged "
        "from the emergency ward at 09:12.",
    ],
    "REC-002": [
        "A patient record was opened at 22:40 after the individual "
        "experienced seizures 3 times in a row. The patient was stabilized "
        "within 18 minutes.",
        "A patient record was opened at 22:40 after the individual "
        "experienced seizures 3 times in a row. The patient was stabilized "
        "within 18 minutes and monitored for 96 hours.",
    ],
    "REC-003": [
        "During the 12 June 2024 admission, patient T.L. (MRN redacted) "
        "experienced a vasovagal episode lasting 210 seconds, monitored "
        "across 4 vital-sign channels. Symptoms cleared without "
        "intervention after 35 minutes.",
        "During the 12 June 2024 admission, the only patient on the ward "
        "being treated for cystic fibrosis experienced a vasovagal episode "
        "lasting 210 seconds, monitored across 4 vital-sign channels. "
        "Symptoms cleared without intervention after 35 minutes.",
        "During the 12 June 2024 admission, a patient experienced a "
        "vasovagal episode lasting 210 seconds, monitored across 4 "
        "vital-sign channels. Symptoms cleared without intervention after "
        "35 minutes.",
    ],
    "REC-004": [
        "At 03:05, the patient's monitor triggered an infusion alarm. A "
        "nurse acknowledged the alert within 2 minutes. A miscalibrated "
        "pump had delivered 18,400 mcg of medication over 26 minutes. The "
        "patient ID appears in the medication administration record next "
        "to each dose.",
    ],
}

rec1 = RECORDS["REC-001"]
print("RECORD REC-001:")
print(textwrap.fill(rec1.text, width=98))
print()
print("must NOT survive:", rec1.identifiers)
print("must survive:    ", rec1.clinical_facts)

We have a shortcut here that a real deployment usually wouldn't: **we already know the ground truth**, the correct answer for every record. The clinical team that filed the record already knows. On the other hand, a real deployment doesn't hand you that answer key.

## 1.1 Why not just ask an LLM?

The obvious way to check a de-identification is to send the before/after to an AI model and ask [*"did this keep the facts and drop the names?"*](https://arxiv.org/abs/2306.05685) But that costs money (per check), isn't exact (it has its own error rate) and it is [self-checking in the worst sense](https://arxiv.org/abs/2310.01798): the model grading the work is the same kind of model that wrote it.

Research on training models to check their own answers ([Liu et al.'s 2025 NeurIPS paper, "Trust, But Verify"](https://neurips.cc/virtual/2025/loc/san-diego/poster/116768)) has a name for exactly this failure: "superficial self-reflection." The model looks like it's double-checking, without actually catching its own mistakes. Now compare: checking whether Costa's MRN, `88431`, survived a rewrite isn't a judgment call. It's a question a computer can answer exactly: is this string present in that text?

## 1.2 The idea in one line

> If you can decide the answer from information already at hand, code can decide it too.

This is the premise behind [*verifiable reward*](https://arxiv.org/abs/2411.15124): replace a learned or human-supplied reward with the output of a program, so the training signal inherits the program's exactness, along with its blind spots, as you'll see later.

We know the identifiers, we know the facts. Both rules of this task reduce to one operation: is this string present in that text?

The [Safe Harbor rule](https://www.hhs.gov/hipaa/for-professionals/special-topics/de-identification/index.html) under HIPAA (the U.S. law governing patient-data privacy) spells out eighteen categories of identifiers that have to disappear before data counts as de-identified. It's the same fixed, public list Costa's redaction has to satisfy. This is the same idea behind *verifiable reward* in current AI research. Training a model, in short, means repeatedly adjusting it so it does more of whatever scores well and less of whatever doesn't; a *reward* is just the number that scoring produces. If a check can run as code, that number can come directly from the check, instead of from another model's opinion.

## 2. Your turn: build the judge

A verdict alone, pass or fail, can stop a loop. It can't drive one: the AI model needs to know exactly which rule it broke, in words specific enough to act on. So the judge you're about to build returns both: pass/fail, and if it fails, why.

You'll test it on a real attempt the model made at de-identifying Mariana Costa's record (`RECORDS["REC-001"]`), and this attempt is broken on purpose. It still leaks something it shouldn't. Two small pieces, one at a time: first the rule for identifiers, then the rule for facts. Whatever you pick for each becomes the actual rule `verify` runs on every record for the rest of the notebook, not a one-off answer just for this example.

### 2.1 TO DO 1: Does the identifier survive?

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

widgets.Widget.close_all()

attempt = RECORDED_LLM_ATTEMPTS["REC-001"][0]

TODO1_OPTIONS = {
    "a rule should mark an identifier as leaked when that identifier shows up somewhere in the text": lambda ident, text: ident in text,
    "a rule should mark an identifier as leaked when the entire text fits inside that identifier": lambda ident, text: text in ident,
    "a rule should mark an identifier as leaked only when that identifier matches the entire text, word for word": lambda ident, text: ident == text,
}
CORRECT_TODO1 = "a rule should mark an identifier as leaked when that identifier shows up somewhere in the text"

attempt_box = widgets.HTML(
    "<b>model's de-identification attempt (RECORDED_LLM_ATTEMPT REC-001):</b><br>{}<br><br>"
    "<b>Identifiers</b> (must NOT survive): <code>Mariana Costa</code>, <code>Costa</code>, <code>88431</code><br><br>"
    "Which rule correctly defines an identifier leak?<br><br>".format(attempt)
)

todo1_pick = widgets.RadioButtons(
    options=list(TODO1_OPTIONS), value=None,
    description="TODO 1:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="max-content"))


def _todo1_feedback(pick):
    if pick is None:
        return
    print()
    caught = TODO1_OPTIONS[pick]("Costa", attempt)
    print("Applying that rule to 'Costa':", "yes, it's still there" if caught else "no, it's gone")
    if pick == CORRECT_TODO1:
        display(HTML("<span style='color:#2e7d32'>correct rule</span>"))
    else:
        display(HTML("<span style='color:#b71c1c'>not quite: try a different option</span>"))


todo1_result = widgets.interactive_output(_todo1_feedback, {"pick": todo1_pick})

display(widgets.VBox([attempt_box, todo1_pick, todo1_result]))

Good, `Costa` gets caught. What you just defined isn't about Costa. It's a general rule: does any identifier show up anywhere inside a piece of text? If yes, that's a leak. That's the "identifier" half of your judge, done. In technical terms: you just implemented a deterministic verifier, a mechanical check that catches identifier leakage before it passes as clean text. Now the second piece.

### 2.2 TO DO 2: Does a fact survive?

In [ ]:
widgets.Widget.close_all()

TODO2_OPTIONS = {
    "a rule should mark a fact as lost when that fact is present in the text": lambda fact, text: fact in text,
    "a rule should mark a fact as lost only when that fact is missing from the text": lambda fact, text: fact not in text,
    "a rule should mark a fact as lost only when the entire text is empty": lambda fact, text: not text,
}
CORRECT_TODO2 = "a rule should mark a fact as lost only when that fact is missing from the text"

fact_box = widgets.HTML(
    "<b>same attempt as before (RECORDED_LLM_ATTEMPT REC-001):</b><br>{}<br><br>"
    "<b>Clinical facts</b> (must survive): <code>14 March 2024</code>, <code>47 minutes</code>, <code>1204 mL</code>, <code>09:12</code><br><br>"
    "Does any of the clinical facts above go missing from it? No, \"47 minutes\" is "
    "still there. The correct rule should say the same.<br><br>".format(attempt)
)

todo2_pick = widgets.RadioButtons(
    options=list(TODO2_OPTIONS), value=None,
    description="TODO 2:", style={"description_width": "70px"},
    layout=widgets.Layout(width="max-content"))


def _todo2_feedback(pick):
    if pick is None:
        return
    print()
    lost = TODO2_OPTIONS[pick]("47 minutes", attempt)
    print("Applying that rule to '47 minutes':", "yes, it's lost" if lost else "no, it's still there")
    if pick == CORRECT_TODO2:
        display(HTML("<span style='color:#2e7d32'>correct rule</span>"))
    else:
        display(HTML("<span style='color:#b71c1c'>not quite: try a different option</span>"))


todo2_result = widgets.interactive_output(_todo2_feedback, {"pick": todo2_pick})

display(widgets.VBox([fact_box, todo2_pick, todo2_result]))

`47 minutes` survives, as expected. The rule you picked generalizes beyond this one string: it checks whether any clinical fact is missing from the text and flags it when one is. That completes the fact side of your judge.

### 2.3 Put both pieces together

You've now defined both halves of your deterministic verifier: one rule that blocks identifier leakage, one that blocks fact loss. Below, they're combined.

`Result` is a small container for a verdict: `passed` (True/False), which `rule` broke it if any, and a human-readable `detail`. Its `__repr__` is just what makes printing one look clean. And `verify` is the function that runs your two picks against the attempt above, one after another, returning the first failure it finds (or PASS, if both hold). The last line calls it and prints what comes back.

`check1` and `check2` inside `verify` are your TO DO 1 and TO DO 2 picks, looked up and called directly. Whatever you selected above is what actually runs below.

Remember what you're aiming for here: this attempt is broken, so **FAIL is the win**. Run the cell below.

In [ ]:
import re

RED   = "\033[41m\033[97m{}\033[0m"   # the verifier caught this
AMBER = "\033[43m\033[30m{}\033[0m"   # this passed, and it should not have


def mark(wrapped: str, phrase: str, template: str) -> str:
    """Highlight `phrase` inside already-wrapped text, tolerating line breaks."""
    pattern = r"\s+".join(re.escape(word) for word in phrase.split())
    return re.sub(pattern, lambda m: template.format(m.group(0)), wrapped)


class Result(NamedTuple):
    passed: bool
    rule: Optional[str] = None      # "identifier_survived" | "fact_lost"
    detail: str = ""                # human-readable reason, filled in only on failure

    def __repr__(self):
        # controls what print(result) actually shows
        if self.passed:
            return "\033[32m\u2713 PASS\033[0m - every check satisfied"
        return "\033[31m\u2717 FAIL\033[0m -- {}: {}".format(self.rule, self.detail)


def verify(text: str, identifiers: List[str], clinical_facts: List[str]) -> Result:
    if todo1_pick.value is None or todo2_pick.value is None:
        raise RuntimeError("Pick an option for both TODO 1 and TODO 2 above before running this.")
    if todo1_pick.value != CORRECT_TODO1 or todo2_pick.value != CORRECT_TODO2:
        raise RuntimeError("TODO 1 and/or TODO 2 above is set to the wrong option: fix that pick before running this.")

    check1 = TODO1_OPTIONS[todo1_pick.value]  # your TODO 1 pick, turned into the actual function that runs
    for ident in identifiers:
        if check1(ident, text):
            return Result(False, "identifier_survived", "found {!r}".format(ident))  # stop at the first leak found

    check2 = TODO2_OPTIONS[todo2_pick.value]  # your TODO 2 pick, turned into the actual function that runs
    for fact in clinical_facts:
        if check2(fact, text):
            return Result(False, "fact_lost", "missing {!r}".format(fact))  # stop at the first missing fact

    return Result(True)  # reached only if every identifier and every fact checked out


result = verify(attempt, rec1.identifiers, rec1.clinical_facts)

print("RECORDED_LLM_ATTEMPT REC-001")
if not result.passed:
    found = result.detail.split("'")[1]  # pull the exact string out of "found 'Costa'" / "missing '47 minutes'"
    print(textwrap.fill(attempt, width=98).replace(found, RED.format(found)))
else:
    print(textwrap.fill(attempt, width=98))

print()
print(result)

It's **FAIL**, and that's the win. `Costa` is still sitting there, one line down from where her full name was removed. A judge that said PASS here would have missed a real leak; yours didn't. `verify` stopped at the identifier rule and never reached the fact rule. The next example is where that one gets tested.

Now a harder one. Here's a second record, already fully stripped of every identifier. But remember, this attempt is broken too. Run the cell below.

In [ ]:
rec2 = RECORDS["REC-002"]
attempt2 = RECORDED_LLM_ATTEMPTS["REC-002"][0]
print("RECORD REC-002:")
print(textwrap.fill(rec2.text, width=98))
print()
print("RECORDED_LLM_ATTEMPT REC-002:")
print(textwrap.fill(attempt2, width=98))

**Question**: Run your verifier on the REC-002 attempt above: **PASS** or **FAIL**? As with REC-001, PASS means neither rule was broken; FAIL means at least one was.

In [ ]:
print(verify(attempt2, rec2.identifiers, rec2.clinical_facts))

It fails too, but for a completely different reason. Read it again: the de-identification worked so well it took `96 hours` (how long the patient was monitored) with it, so the record is safe but no longer usable for the study. That second failure is the one that's easy to miss. Most of what a verifier is worth lives in the constraint you almost forgot to write down.

## 3. Closing the loop

Back in "Why not just ask an LLM?" the complaint was that a judge only grades; it never gives the generator anything to act on. A verdict with a reason fixes that: the generator gets something to act on, not just a stop sign. Feed the reason back, let the model try again, check again, and stop when it passes or when you run out of patience. A repair loop with no cap on attempts brings back the same cost problem: you can spend money indefinitely.

Here's the shape of it:

<div style="text-align:center; margin: 22px 0;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 200" style="max-width:680px; width:100%; font-family: sans-serif;">
<defs>
<marker id="ag" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#5b5f66"></path></marker>
<marker id="ar" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#5b5f66"></path></marker>
<filter id="softShadow" x="-30%" y="-30%" width="160%" height="160%">
<feDropShadow dx="0" dy="6" stdDeviation="6" flood-color="#1a1a1a" flood-opacity="0.18"></feDropShadow>
</filter>
<path id="loopPath" d="M85,88 L655,88 L655,168 L85,168 Z" fill="none"></path>
</defs>
<rect x="10" y="60" width="150" height="56" rx="12" fill="#fff" stroke="#d9dce0"></rect>
<text x="85" y="93" text-anchor="middle" font-size="13" fill="#1a1a1a">model output</text>
<rect x="200" y="60" width="150" height="56" rx="12" fill="#fff" stroke="#d9dce0"></rect>
<text x="275" y="93" text-anchor="middle" font-size="13" fill="#1a1a1a">verifier</text>
<rect x="390" y="60" width="160" height="56" rx="12" fill="#fff" stroke="#d9dce0"></rect>
<text x="470" y="83" text-anchor="middle" font-size="13" fill="#1a1a1a">structured</text>
<text x="470" y="100" text-anchor="middle" font-size="13" fill="#1a1a1a">feedback</text>
<rect x="600" y="60" width="110" height="56" rx="12" fill="#fff" stroke="#d9dce0"></rect>
<text x="655" y="93" text-anchor="middle" font-size="13" fill="#1a1a1a">retry</text>
<line x1="160" y1="88" x2="196" y2="88" stroke="#5b5f66" stroke-width="2" marker-end="url(#ag)"></line>
<line x1="350" y1="88" x2="386" y2="88" stroke="#5b5f66" stroke-width="2" marker-end="url(#ag)"></line>
<line x1="550" y1="88" x2="596" y2="88" stroke="#5b5f66" stroke-width="2" marker-end="url(#ag)"></line>
<path d="M655,116 L655,168 L85,168 L85,116" fill="none" stroke="#5b5f66" stroke-width="2" stroke-dasharray="7,5" marker-end="url(#ar)"></path>
<text x="370" y="188" text-anchor="middle" font-size="12.5" fill="#1a1a1a">until it passes, or the attempt cap is hit</text>
</svg>
</div>

**Figure 2.** The shape the code below runs. A program holds the gate, not a judge: the reason travels with the retry.

The code below runs that loop on the two records that just failed for different reasons: REC-001 (Costa's name resurfacing) and REC-002 (a 96-hour gap in the record). Watch what gets printed for each attempt: pass, fail, and why. Run the cell below.

In [ ]:
class RecordedGenerator:
    # Replays recorded attempts, one per call, mimicking a repair loop.
    # The verifier's feedback is accepted and ignored: a recording cannot
    # react to it. That is a real limitation of teaching this offline. The
    # shape of the loop is faithful, the responsiveness is staged.

    def __init__(self, record_id: str):
        self._attempts = RECORDED_LLM_ATTEMPTS[record_id]  # the fixed sequence to replay
        self._calls = 0  # how many attempts already handed out

    def __call__(self, feedback: str = "") -> str:
        if self._calls >= len(self._attempts):
            raise RuntimeError("No recorded attempt left for this record.")
        attempt = self._attempts[self._calls]  # next attempt in the recording
        self._calls += 1
        return attempt


def repair_loop(record: Record, max_iters: int = 3, generator=None):
    # Returns (final_text, result, n_attempts)
    if generator is None:
        generator = RecordedGenerator(record.record_id)
    feedback = ""  # nothing to report back yet: this is the first try
    for i in range(max_iters):
        text = generator(feedback)  # generate (here, replay) the next attempt
        result = verify(text, record.identifiers, record.clinical_facts)  # verify it
        print("\033[1m  attempt {}:\033[0m".format(i + 1))  # show the attempt as it happens, verdict last
        wrapped = textwrap.fill(text, width=90, initial_indent="    ", subsequent_indent="    ")
        if not result.passed:
            found = result.detail.split("'")[1]  # pull the exact string out of "found '...'" / "missing '...'"
            wrapped = wrapped.replace(found, RED.format(found))  # highlight it, same as before
        print(wrapped)
        print("  {}".format(result))
        if result.passed:
            return text, result, i + 1  # stop here and record how many tries it took
        feedback = "Rule broken: {}. {}".format(result.rule, result.detail)  # the reason, fed to the next attempt
        print()  # separate this failed attempt from the next one
    return text, result, max_iters  # ran out of attempts, return the last one anyway


for rid in ["REC-001", "REC-002"]:
    record = RECORDS[rid]
    print("\033[1m{}\033[0m".format(rid))
    print()
    original_line = textwrap.fill("  original: " + record.text, width=90, subsequent_indent="    ")
    original_line = original_line.replace("original:", "\033[1moriginal:\033[0m", 1)  # bold the label after wrapping, so it does not skew the width
    print(original_line)
    print()
    text, result, n = repair_loop(record)
    print()
    print("  -> passed={} after {} attempt(s)".format(result.passed, n))
    print()

About that `RecordedGenerator`: it's a stand-in for the model, and what matters is what it does, not how. It hands back one recorded attempt per call and accepts the feedback without reading it, which is the caveat at the bottom of this cell.

That's the loop attempt by attempt: `REC-001` fails twice, first for the name, then for the MRN, before a clean pass on the third try. `REC-002` fails once missing the `96 hours`, before a clean pass on the second.

This shape, try, check, explain what broke, try again, is also the mechanism behind training a model with verifiable rewards: score many attempts with a check like this one, then nudge the model toward whatever scores well. You're about to build the small, by-hand version of the same signal that trains systems like [DeepSeek-R1](https://www.nature.com/articles/s41586-025-09422-z), whose 2025 paper, "DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning," describes this training approach directly.

One honest note, worth restating now that it matters: you already knew `RECORDED_LLM_ATTEMPTS` was a set of fixed recordings, not a live model. That was true when you first read Mariana Costa's record. It didn't matter then because you were only reading a finished attempt. It matters now: `RecordedGenerator` can't actually read the feedback you feed it, so it replays what a real attempt sequence looked like rather than reacting to it. That's the one part of this loop you shouldn't read as faithful.

## 4. Now the interesting part

You've closed the loop: generate, verify, explain, retry, until it passes. That loop has two names: generating several attempts and keeping only the ones that pass is [rejection sampling](https://arxiv.org/abs/2307.09288); feeding the verifier's reason for failure back into the next attempt is [self-correction](https://aclanthology.org/2024.tacl-1.27/). The question now is what "it passes" actually means. Run your verifier over every attempt the model made at de-identifying a third record. All three pass.

Before you move on:

1. Read each attempt carefully;
2. Find out if it is still identifying the patient and how.

From here on: **red** is what the verifier caught. **Amber** is what it let through and shouldn't have. Amber is the subject of the rest of this notebook. Run the cell below.

In [ ]:
rec3 = RECORDS["REC-003"]
print("\033[1mRECORD REC-003:\033[0m")
print(textwrap.fill(rec3.text, width=98))
print()
print("\033[1mmust NOT survive:\033[0m", rec3.identifiers)
print("\033[1mmust survive:\033[0m    ", rec3.clinical_facts)
print()

# Read by us, not derived from any verifier: what actually identifies the
# patient in an attempt the verifier waved through.
SILENT_LEAKS = {
    ("REC-003", 0): "T.L.",
    ("REC-003", 1): "the only patient on the ward being treated for cystic fibrosis",
}

for i, att in enumerate(RECORDED_LLM_ATTEMPTS["REC-003"]):
    r = verify(att, rec3.identifiers, rec3.clinical_facts)
    shown = textwrap.fill(att, width=98)
    silent = SILENT_LEAKS.get(("REC-003", i))
    if r.passed and silent:
        shown = mark(shown, silent, AMBER)
    print("\033[1m--- attempt {}\033[0m : passed={}".format(i, r.passed))
    print(shown)
    print()

One of them says *"patient T.L."* Your verifier was looking for `Tomas Lindqvist` and for `Lindqvist`. It found neither, so it passed the text. On a ward with exactly one Lindqvist, `T.L.` identifies him precisely.

Your verifier isn't broken. It did exactly what you told it to do. It just never claimed what you assumed it claimed. This is the dangerous kind of miss: it ships instead of costing you a wasted retry.

<div style="text-align:center; margin: 22px 0;">
<table style="border-collapse:collapse; font-family:sans-serif; font-size:13.5px; margin:0 auto;">
<tr><td style="padding:8px 14px;"></td><td style="padding:8px 14px; font-weight:600; border-bottom:1px solid #999;">verifier flags it</td><td style="padding:8px 14px; font-weight:600; border-bottom:1px solid #999;">verifier clears it</td></tr>
<tr><td style="padding:8px 14px; font-weight:600; border-right:1px solid #999;">actually leaks</td><td style="padding:8px 14px; background:#e6f4ea;">correct catch</td><td style="padding:8px 14px; background:#fdecea;"><b>missed detection</b><br><span style="font-size:12px; color:#b71c1c;">quiet, it ships</span></td></tr>
<tr><td style="padding:8px 14px; font-weight:600; border-right:1px solid #999;">actually clean</td><td style="padding:8px 14px; background:#fff8e1;"><b>false alarm</b><br><span style="font-size:12px; color:#8a6d00;">loud, wastes a retry</span></td><td style="padding:8px 14px; background:#e6f4ea;">correct pass</td></tr>
</table>
</div>

**Figure 3.** The two ways a verifier can be wrong are not the same mistake: one is annoying, the other is the one that ships. We'll call them **loud failures** and **quiet failures** (a false alarm is a [false positive](https://doi.org/10.1016/j.patrec.2005.10.010), a missed detection a false negative), and from here on the whole notebook is about the quiet ones.

Score attempts with a verifier this loose during training, and "ships" becomes "gets rewarded," and nothing in the signal ever points the model away from it.

## 4.1 Extending the judge

Your judge needs two upgrades: catch `Tomas Lindqvist` written as `T.L.`, `T. L.`, or `TL`, and stop confusing thousands separators with missing facts. Before you build either fix, run your original judge on one more record. It's fully de-identified: no name or MRN survives anywhere in it.

**Predict:** does it pass?

In [ ]:
rec4 = RECORDS["REC-004"]
attempt4 = RECORDED_LLM_ATTEMPTS["REC-004"][0]
print("\033[1mRECORD REC-004:\033[0m")
print(textwrap.fill(rec4.text, width=98))
print()
print("\033[1mmust NOT survive:\033[0m", rec4.identifiers)
print("\033[1mmust survive:\033[0m    ", rec4.clinical_facts)
print()
print("\033[1m--- attempt 0\033[0m")
print(textwrap.fill(attempt4, width=98))
print()
print(verify(attempt4, rec4.identifiers, rec4.clinical_facts))

It doesn't. Your judge flags a number, `18400`, as missing. It isn't missing. The model just wrote it `18,400`. The trap is deliberate: a patient ID and a medication dose differ by two but share a four-digit prefix. It's the mirror image of the `T.L.` problem: a false alarm this time, sending perfectly good work back for repair.

Fix both below: strip thousands separators before checking facts, and reject initial forms before passing identifiers on the name side. Watch out: some options are the exact bugs you just met. TO DO 3 decides when an initials-form like `T.L.` counts as leaked; TO DO 4 decides when a fact still counts as present despite a formatting quirk like `18,400` vs `18400`. Whatever you pick becomes the actual rule `verify_v2` runs for the rest of the notebook.

## 4.2 TO DO 3: Does 'T.L.' still leak?

In [ ]:
import re

widgets.Widget.close_all()

attempt_tl = RECORDED_LLM_ATTEMPTS["REC-003"][0]

TODO3_OPTIONS = {
    "an initials-form counts as leaked only when it appears as its own word, not merely as the start of a longer word (a boundary check)": lambda form, text: bool(re.search(r"(?<!\w){}(?!\w)".format(re.escape(form)), text, re.IGNORECASE)),
    "an initials-form counts as leaked whenever it shows up anywhere inside the text, even as part of a longer word": lambda form, text: form in text,
    "an initials-form counts as leaked only when the text starts with it": lambda form, text: text.startswith(form),
}
CORRECT_TODO3 = "an initials-form counts as leaked only when it appears as its own word, not merely as the start of a longer word (a boundary check)"

attempt_box_tl = widgets.HTML(
    "<b>model's de-identification attempt (RECORDED_LLM_ATTEMPT REC-003, attempt 0):</b><br>{}<br><br>"
    "Does 'T.L.', one of Tomas Lindqvist's initials-forms, still show up in it?<br><br>".format(attempt_tl)
)

todo3_pick = widgets.RadioButtons(
    options=list(TODO3_OPTIONS), value=None,
    description="TODO 3:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="max-content"))


def _todo3_feedback(pick):
    if pick is None:
        return
    print()
    caught = TODO3_OPTIONS[pick]("T.L.", attempt_tl)
    print("Applying that rule to 'T.L.':", "yes, it's still there" if caught else "no, it's gone")
    if pick == CORRECT_TODO3:
        display(HTML("<span style='color:#2e7d32'>correct rule</span>"))
    else:
        display(HTML("<span style='color:#b71c1c'>not quite: try a different option</span>"))


todo3_result = widgets.interactive_output(_todo3_feedback, {"pick": todo3_pick})

display(widgets.VBox([attempt_box_tl, todo3_pick, todo3_result]))

## 4.3 TO DO 4: Does '18,400' vs '18400' still count as present?

In [ ]:
widgets.Widget.close_all()

attempt4_raw = RECORDED_LLM_ATTEMPTS["REC-004"][0]

TODO4_OPTIONS = {
    "a fact counts as lost only when it's absent from the text after stripping commas from both fact and text before comparing": lambda fact, text: fact.replace(",", "") not in text.replace(",", ""),
    "a fact counts as lost only when it's absent from the text, compared exactly as written": lambda fact, text: fact not in text,
    "a fact counts as lost when it's present in the text": lambda fact, text: fact in text,
}
CORRECT_TODO4 = "a fact counts as lost only when it's absent from the text after stripping commas from both fact and text before comparing"

attempt_box_fact = widgets.HTML(
    "<b>model's de-identification attempt (RECORDED_LLM_ATTEMPT REC-004, attempt 0):</b><br>{}<br><br>"
    "The model wrote '18,400 mcg'. Does '18400 mcg' still count as present?<br><br>".format(attempt4_raw)
)

todo4_pick = widgets.RadioButtons(
    options=list(TODO4_OPTIONS), value=None,
    description="TODO 4:", style={"description_width": "70px"},
    layout=widgets.Layout(width="max-content"))


def _todo4_feedback(pick):
    if pick is None:
        return
    print()
    lost = TODO4_OPTIONS[pick]("18400 mcg", attempt4_raw)
    print("Applying that rule to '18400 mcg':", "yes, it's lost" if lost else "no, it's still there")
    if pick == CORRECT_TODO4:
        display(HTML("<span style='color:#2e7d32'>correct rule</span>"))
    else:
        display(HTML("<span style='color:#b71c1c'>not quite: try a different option</span>"))


todo4_result = widgets.interactive_output(_todo4_feedback, {"pick": todo4_pick})

display(widgets.VBox([attempt_box_fact, todo4_pick, todo4_result]))

### 4.3.1 Combine the two upgrades

You've picked both halves of the upgrade: which initials-form counts as leaked, and which comparison rule survives the comma quirk. Below is where they get combined into `verify_v2`.

`_initials_of` builds the initials-forms of a name (`Tomas Lindqvist` becomes `T.L.`, `T. L.`, `TL`). `verify_v2` keeps the structure of `verify`, now also checking initials and stripping commas before comparing facts. `check3` and `check4` inside it are the TO DO 3 and TO DO 4 options you picked, looked up and called directly.

The loop at the bottom re-runs `verify` (v1) and `verify_v2` (v2) side by side on attempts you've already seen fail. Watch for where v2 disagrees with v1.

In [ ]:
def _initials_of(name: str) -> List[str]:
    """Three initials-forms of a full name: 'Tomas Lindqvist' -> T.L., T. L., TL."""
    parts = name.split()
    if len(parts) < 2:
        return []
    letters = [p[0].upper() for p in parts]
    return [".".join(letters) + ".", ". ".join(letters) + ".", "".join(letters)]


def verify_v2(text: str, identifiers: List[str], clinical_facts: List[str]) -> Result:
    if todo3_pick.value is None or todo4_pick.value is None:
        raise RuntimeError("Pick an option for both TODO 3 and TODO 4 above before running this.")
    if todo3_pick.value != CORRECT_TODO3 or todo4_pick.value != CORRECT_TODO4:
        raise RuntimeError("TODO 3 and/or TODO 4 above is set to the wrong option: fix that pick before running this.")

    check3 = TODO3_OPTIONS[todo3_pick.value]  # your TODO 3 pick, turned into the actual function that runs
    check4 = TODO4_OPTIONS[todo4_pick.value]  # your TODO 4 pick, turned into the actual function that runs

    norm_text = text.replace(",", "")  # 18,400 -> 18400
    for ident in identifiers:
        if ident in norm_text:
            return Result(False, "identifier_survived", "found {!r}".format(ident))
        for form in _initials_of(ident):
            if check3(form, norm_text):
                return Result(False, "identifier_survived", "initials {!r}".format(form))
    for fact in clinical_facts:
        if check4(fact, norm_text):
            return Result(False, "fact_lost", "missing {!r}".format(fact))
    return Result(True)


for rid, record in [("REC-003", rec3), ("REC-004", rec4)]:
    print("\033[1m{}\033[0m".format(rid))
    print()
    original_line = textwrap.fill("  original: " + record.text, width=90, subsequent_indent="    ")
    original_line = original_line.replace("original:", "\033[1moriginal:\033[0m", 1)
    print(original_line)
    print()
    for i, att in enumerate(RECORDED_LLM_ATTEMPTS[rid]):
        print("\033[1m  attempt {}:\033[0m".format(i))
        print(textwrap.fill(att, width=90, initial_indent="    ", subsequent_indent="    "))
        print("  v1: {}".format(verify(att, record.identifiers, record.clinical_facts)))
        print("  v2: {}".format(verify_v2(att, record.identifiers, record.clinical_facts)))
        print()

`T.L.` no longer slips through, and the `18,400` vs `18400` quirk no longer costs you a false alarm. `verify_v2` says something true that `verify` could not.

Attempt 1, though, remains a quiet failure under both versions of your verifier. Extending a verifier closes the gaps you looked for, nothing more. That's true of any patched filter or guardrail, not just this one, and it's the gap you're about to walk into.

## 5. The fix has a blind spot too

Before moving on: run your improved verifier on two sentences a model might write about entirely different patients, sentences that have nothing to do with these records:

**Sentence 1:** "ASA 325 mg was administered for pain relief."
- Context: this identifier belongs to a patient named "Ana Silva", who is not mentioned in the sentence.
- must NOT survive: `['Ana Silva']`
- must survive: `['325 mg']`

**Sentence 2:** "Transferred to ICU for monitoring."
- Context: this identifier belongs to a patient named "Isabel Cortez Uribe", who is not mentioned in the sentence.
- must NOT survive: `['Isabel Cortez Uribe']`
- must survive: `[ ]`

**Question:** does either one wrongly fail? Run the cell below.

In [ ]:
sentence_asa = "ASA 325 mg was administered for pain relief."
print("sentence 1:", verify_v2(sentence_asa, ["Ana Silva"], ["325 mg"]))
sentence_icu = "Transferred to ICU for monitoring."
print("sentence 2:", verify_v2(sentence_icu, ["Isabel Cortez Uribe"], []))

Your boundary check already handles `ASA` correctly. `AS` no longer matches as a prefix, so this sentence passes clean with no false alarm. But **`ICU` still fails.** Not because of a boundary bug: `ICU` is a common acronym that happens to match the initials of `Isabel Cortez Uribe`, a different patient. No boundary check can tell an acronym from a coincidence; the string itself is genuinely ambiguous.

## 5.1 Auditing on purpose, instead of waiting to get unlucky

That `ASA`/`ICU` bug wasn't found by luck. Someone went looking for it using a checklist that generalizes to any string-matching predicate: casing and whitespace, plus a mirror question: does clean text get wrongly rejected?

**Your audit checklist.** Run this against any string-matching rule you write. Each line is a way two strings can mean the same thing, or mean different things and look identical.

- **casing**: `costa` vs `Costa`
- **whitespace and punctuation**: `T.L.` / `T. L.` / `TL`
- **look-alike characters**: Cyrillic `а` is not Latin `a`
- **shortened forms**: initials, nicknames, truncations
- **number formatting**: `18,400` / `18400` / `18 400`
- **substring collisions**: `ASA`, `ICU`: the acronym that is also an identifier
- **containment is not equality**: `12045 mL` contains the correct `1204 mL`
- **the mirror question**: *does clean text get wrongly rejected?*

Two of these you have already met, in the initials-form and comma-formatting fixes. Casing and look-alike characters never came up above. Keep them on the list anyway; a check that has never been tested against them is not the same as a check that has passed. The rest, this notebook is about to find.

Try this one before running it: a model writes a note that mentions "early-onset Parkinson disease." The identifier list (from a completely different patient) includes the surname `Parkinson`. Nothing here was actually leaked; the sentence is about a diagnosis, not that patient.

**Predict:** does your verifier agree it's clean? Run the cell below.

In [ ]:
sentence_parkinson = "Neurology flagged early-onset Parkinson disease during evaluation."
result = verify_v2(sentence_parkinson, rec2.identifiers, [])
if not result.passed:
    found = result.detail.split("'")[1]  # pull the exact string out of "found '...'"
    print("Sentence:", sentence_parkinson.replace(found, RED.format(found)))
else:
    print(sentence_parkinson)
print(result)

No, the sentence fails. Telling Parkinson the patient from Parkinson the diagnosis apart is [named entity disambiguation](https://aclanthology.org/E06-1002/), not a quick fix in code, so it gets written down as a known gap, not swept under the rug. That distinction, fixing what's bounded and mechanical while documenting what's genuinely open-ended, is worth more than any specific bug. The fact side needed the same audit: a corrupted value like `12045 mL` still contains the correct `1204 mL`, so a containment-based check would call the fact present when the number is actually wrong.

## 5.2 The part that doesn't have a fix

Your extended verifier now catches `T.L.` Here is the model's second attempt at de-identifying that same record: REC-003.

**Question:** does it catch this one too? Run the cell below.

In [ ]:
cystic_fibrosis_attempt = RECORDED_LLM_ATTEMPTS["REC-003"][1]
print("\033[1mREC-003, attempt 1:\033[0m")
print()
print(textwrap.fill(cystic_fibrosis_attempt, width=98))
print()
result = verify_v2(cystic_fibrosis_attempt, rec3.identifiers, rec3.clinical_facts)
print(result)
if result.passed:
    print()
    print(mark(textwrap.fill(cystic_fibrosis_attempt, width=98), SILENT_LEAKS[("REC-003", 1)], AMBER))

Read it again: *"the only patient on the ward being treated for cystic fibrosis..."* No name. No initials. No MRN. Zero identifying strings, yet it identifies one person exactly to anyone who knows the ward.

This is a different kind of gap from the `T.L.` fix a few sections back. While `T.L.` was a fixed, computable transformation of a string (`Tomas Lindqvist`), this phrase isn't a transformation of anything. Knowing that it identifies him requires facts (who's on the ward and what they're being treated for) that live nowhere in your input. Not in the text, not in your identifier list, not in your facts list. No amount of string matching gets you there because the input doesn't contain what you'd need to answer the question.

Two names worth keeping. `T.L.` was a **string gap**: a fixed, computable transformation of something you already hold. Widen the rule and you close it. *"The only patient on the ward being treated for cystic fibrosis"* is a **world gap**: nothing connects that phrase to Tomas Lindqvist except a fact that lives outside your input. No rule closes it because there is nothing in the input for a rule to look at. Re-identifying someone from a handful of ordinary-looking details is a documented problem in health-data privacy research; [*k-anonymity*](https://doi.org/10.1142/S0218488502001648) asks how many people share this exact combination of details.

Ask an LLM to read this sentence with the same context a human has, and it might catch what no string rule can, trading exactness and cost for semantic reach. The instinct is to add a third rule, but training against a verifier with blind spots means whatever it can't see is exactly what gets exploited, a preview of the reward hacking ahead. What would the fourth gap be? These records were built with traps we knew about; a real corpus hands you formats nobody anticipated, forever. Try the check below before reading on.

In [ ]:
widgets.Widget.close_all()

GAP_OPTIONS = {
    "\"Patient MRN 88431 was treated for anaphylaxis.\"": "matches an identifier directly: verify_v2 already catches this.",
    "\"The physician remembers this as the case with the unusual dosing error.\"": "no name, no MRN, no initials. But it points at one specific case using something that isn't in the text, the identifier list, or the facts list at all.",
    "\"Patient T.L. was monitored for four hours.\"": "an initials-form: verify_v2's boundary check already catches this one.",
}
CORRECT_GAP = "\"The physician remembers this as the case with the unusual dosing error.\""

gap_box = widgets.HTML(
    "Which of these would still identify a patient, no matter how many "
    "string-matching rules you add to <code>verify_v2</code>?<br><br>"
)

gap_pick = widgets.RadioButtons(
    options=list(GAP_OPTIONS), value=None,
    description="Pick:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="max-content"))


def _gap_feedback(pick):
    if pick is None:
        return
    print()
    print(GAP_OPTIONS[pick])
    if pick == CORRECT_GAP:
        display(HTML("<span style='color:#2e7d32'>Right. Same shape of gap as the cystic-fibrosis case: no string to match, just outside knowledge.</span>"))
    else:
        display(HTML("<span style='color:#b71c1c'>Not this one. String matching already handles it. Try the option with no name, MRN, or initials-form in it at all.</span>"))


gap_result = widgets.interactive_output(_gap_feedback, {"pick": gap_pick})

display(widgets.VBox([gap_box, gap_pick, gap_result]))

## 6. What it takes to close that gap

Watch what happens when the input grows by one fact: a new piece of information, a ward diagnosis log.

`WARD_DIAGNOSIS_LOG` is the new piece of information, with one diagnosis per patient. `flags_by_unique_diagnosis` counts how many patients on the log share each diagnosis, then checks whether the text says "the only patient" and names a diagnosis that only one patient has. If both hold, that diagnosis points to exactly one name, and the function returns it. Run the cell below.

In [ ]:
from collections import Counter

WARD_DIAGNOSIS_LOG = {
    "Tomas Lindqvist": "cystic fibrosis",
    "Priya Parkinson": "epilepsy",
    "Diego Fernandez": "hypertension",
    "Wei Zhang": "asthma",
}

def flags_by_unique_diagnosis(text: str, diagnosis_log: dict) -> Optional[str]:
    """Return the patient a "the only patient ... <diagnosis>" phrase would
    identify, if any.

    Decidable now for exactly the reason it was not decidable before: the
    fact that makes the phrase identifying, which diagnosis is unique on
    the ward, is now part of the input, not just part of the world.
    """
    counts = Counter(diagnosis_log.values())
    for name, dx in diagnosis_log.items():
        if counts[dx] == 1 and "only patient" in text.lower() and dx.lower() in text.lower():
            return name
    return None

print("\033[1mREC-003, attempt 1 (same as before):\033[0m")
print()
print(textwrap.fill(cystic_fibrosis_attempt, width=98))
print()
print("Identified patient:", flags_by_unique_diagnosis(cystic_fibrosis_attempt, WARD_DIAGNOSIS_LOG))

Supplying the log didn't make string matching cleverer. It did the one thing that ever restores decidability: **moving the fact inside**, out of *the world* and into *the input*. A technique called [retrieval-augmented generation (RAG)](https://arxiv.org/abs/2005.11401) makes the same move: instead of hoping a model already knows something, you fetch it and hand it over as context. Generation and verification hit the same wall. Moving the fact inside isn't a general fix: it closes the one phrasing you anticipated and needs a new fact for every category a description might lean on. The next phrasing will need its own log.

## 6.1 The claim a verifier supports

<div style="display:flex; gap:18px; flex-wrap:wrap; margin:18px 0;">
<div style="flex:1 1 260px; border:1px solid #bbb; padding:14px 18px; background:#e6f4ea;">
<div style="font-size:12px; text-transform:uppercase; letter-spacing:.06em; color:#1b5e20;">The claim a verifier supports</div>
<div style="font-size:16px; margin-top:6px;">No output contains a leak of a form I checked for.</div>
</div>
<div style="flex:1 1 260px; border:1px solid #bbb; padding:14px 18px; background:#fdecea;">
<div style="font-size:12px; text-transform:uppercase; letter-spacing:.06em; color:#b71c1c;">The claim people hear</div>
<div style="font-size:16px; margin-top:6px;">No output leaks.</div>
</div>
</div>

The first is a fact you can stand behind. The second is a hope. The distance between them is every string gap you never widened the rule for, and every world gap you never could. None of this makes verifiers a bad idea. Next to a model acting as judge, a deterministic gate is still exact where it applies, and honest about where it doesn't, at no cost per run. What changes is the sentence you're actually entitled to write afterward.

In theorem proving, this same idea gets pushed all the way. A proof assistant (software like [Lean](https://doi.org/10.1007/978-3-030-79876-5_37) or Coq that mechanically checks whether each step really follows from the stated axioms) has existed for decades; [DeepSeek-Prover](https://arxiv.org/abs/2408.08152)'s contribution is training an LLM to generate proofs at the scale a proof assistant like these can actually accept, using the same generate-verify-reward loop as DeepSeek-R1, but for mathematics instead of free text.

When a proof assistant says a proof checks out, that guarantee really is total: a formal proof doesn't have the kind of open-ended, anything-goes input space free text does. The de-identification verifier's partial coverage isn't a shortcoming of this particular verifier; it's what checking free text costs you. The interesting failure here isn't a verifier being *wrong*. It's a verifier being *right about less than you assumed*, reported in the same tone of voice it uses for everything else: every way a record can still identify a patient, or quietly lose a fact.

<div style="text-align:center; margin: 22px 0;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 34 660 290" style="max-width:660px; width:100%; font-family: sans-serif;">
<rect x="26" y="60" width="304" height="240" rx="0" fill="#1b5e20"></rect>
<rect x="338" y="60" width="296" height="240" rx="0" fill="#b71c1c"></rect><text x="178" y="86" text-anchor="middle" font-size="12.5" font-weight="600" fill="#ffffff">a PASS from verify_v2 rules this out</text>
<text x="44" y="118" font-size="11.5" fill="#ffffff">✓  an identifier present, written exactly</text>
<text x="58" y="133" font-size="10" fill="#ffffff" font-family="'IBM Plex Mono',monospace">"Costa", "88431"   ·   v1</text>
<text x="44" y="160" font-size="11.5" fill="#ffffff">✓  a listed fact missing outright</text>
<text x="58" y="175" font-size="10" fill="#ffffff" font-family="'IBM Plex Mono',monospace">"96 hours" dropped   ·   v1</text>
<text x="44" y="202" font-size="11.5" fill="#ffffff">✓  an initials-form of a listed name</text>
<text x="58" y="217" font-size="10" fill="#ffffff" font-family="'IBM Plex Mono',monospace">"T.L.", "T. L.", "TL"   ·   v2</text>
<text x="44" y="244" font-size="11.5" fill="#ffffff">✓  a fact hidden by comma formatting</text>
<text x="58" y="259" font-size="10" fill="#ffffff" font-family="'IBM Plex Mono',monospace">"18,400" = "18400"   ·   v2</text>
<text x="486" y="86" text-anchor="middle" font-size="12.5" font-weight="600" fill="#ffffff">a PASS says nothing about this</text>
<text x="352" y="118" font-size="11.5" fill="#ffffff">✗  world gaps</text>
<text x="366" y="133" font-size="10" fill="#ffffff" font-family="'IBM Plex Mono',monospace">"the only patient … cystic fibrosis"</text>
<text x="352" y="160" font-size="11.5" fill="#ffffff">✗  entity ambiguity</text>
<text x="366" y="175" font-size="10" fill="#ffffff" font-family="'IBM Plex Mono',monospace">"Parkinson" the diagnosis, not the patient</text>
<text x="352" y="202" font-size="11.5" fill="#ffffff">✗  containment ≠ equality</text>
<text x="366" y="217" font-size="10" fill="#ffffff" font-family="'IBM Plex Mono',monospace">"12045 mL" contains the correct "1204 mL"</text>
<text x="352" y="244" font-size="11.5" fill="#ffffff">✗  any form nobody listed</text>
<text x="366" y="259" font-size="10" fill="#ffffff" font-family="'IBM Plex Mono',monospace">nicknames, misspellings, a 19th identifier</text>
</svg>
</div>

**Figure 4.** What a PASS is entitled to claim. Inside the shaded region, a PASS is a fact. Outside it, a PASS is silence. Extending the verifier grows the green box; it never removes the boundary.

## 6.2 What happens when you train against this

So far, you've used the verifier by hand: read the feedback, pick the next attempt. Training against a verifiable reward does something different. It scores many attempts and nudges a model toward whatever scores well. Nothing here trains anything, but the same recordings let you compute the number that tells you whether it would work.

Three attempts are about to get scored under your extended verifier: the one with `T.L.` (already caught by v2's boundary check), the one naming *"the only patient ... cystic fibrosis"* (which leaks: a human reading it identifies `Tomas Lindqvist`, and your verifier still can't catch it), and the fully clean one (which doesn't leak at all).

**Question:** does a verifier that scores each attempt reward the leaking one any less than the clean one? Run the cell below.

In [ ]:
from IPython.display import HTML, display

leaks_on_inspection = [True, True, False]  # read by us; not derived from any verifier

rows = []
for i, att in enumerate(RECORDED_LLM_ATTEMPTS["REC-003"]):
    r1 = int(verify(att, rec3.identifiers, rec3.clinical_facts).passed)
    r2 = int(verify_v2(att, rec3.identifiers, rec3.clinical_facts).passed)
    leaks = leaks_on_inspection[i]
    rewarded_anyway = (r2 == 1 and leaks)  # the row this whole section is about
    rows.append(
        "<tr>"
        "<td style='padding:7px 16px'>attempt {}</td>"
        "<td style='padding:7px 16px; text-align:center; color:{}'>{}</td>"
        "<td style='padding:7px 16px; text-align:center'>{}</td>"
        "<td style='padding:7px 16px; text-align:center; background:{}; border:1px solid #ddd'>"
        "<b>{}</b></td>"
        "</tr>".format(
            i,
            "#b71c1c" if leaks else "#1b5e20", "leaks" if leaks else "clean",
            r1,
            "#fdecea" if rewarded_anyway else "#e6f4ea", r2))

display(HTML(
    "<table style='border-collapse:collapse; font-family:sans-serif; font-size:13px; margin:8px 0'>"
    "<tr>"
    "<th style='padding:7px 16px; text-align:left'></th>"
    "<th style='padding:7px 16px'>actually leaks</th>"
    "<th style='padding:7px 16px'>reward (v1)</th>"
    "<th style='padding:7px 16px'>reward (v2)</th>"
    "</tr>" + "".join(rows) + "</table>"))

<div style="text-align:center; margin: 22px 0;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 420 210" style="max-width:380px; width:100%; font-family: sans-serif;">
<line x1="40" y1="160" x2="400" y2="160" stroke="#d9dce0" stroke-width="1"></line>
<rect x="70" y="156" width="70" height="4" fill="#b71c1c"></rect>
<text x="105" y="176" text-anchor="middle" font-size="11" fill="#5b5f66">attempt 0</text>
<text x="105" y="190" text-anchor="middle" font-size="10" fill="#b71c1c">leaks · reward=0</text>
<rect x="180" y="60" width="70" height="100" fill="#b71c1c"></rect>
<text x="215" y="50" text-anchor="middle" font-size="12" font-weight="600" fill="#b71c1c">reward=1</text>
<text x="215" y="176" text-anchor="middle" font-size="11" fill="#5b5f66">attempt 1</text>
<text x="215" y="190" text-anchor="middle" font-size="10" fill="#b71c1c">leaks</text>
<rect x="290" y="60" width="70" height="100" fill="#1b5e20"></rect>
<text x="325" y="50" text-anchor="middle" font-size="12" font-weight="600" fill="#1b5e20">reward=1</text>
<text x="325" y="176" text-anchor="middle" font-size="11" fill="#5b5f66">attempt 2</text>
<text x="325" y="190" text-anchor="middle" font-size="10" fill="#1b5e20">clean</text>
</svg>
</div>

**Figure 5.** Reward under your extended verifier. Attempt 1 leaks and attempt 2 is clean, same bar height. No gradient separates them.

Both attempts score 1 under v2. The quiet failure gets the same reward as the clean one. This has a name: [**reward hacking**](https://proceedings.neurips.cc/paper_files/paper/2022/hash/3d719fee332caa23d5038b8a90e81796-Abstract-Conference.html) (also called [specification gaming](https://deepmind.google/blog/specification-gaming-the-flip-side-of-ai-ingenuity/)), scoring well by exploiting what the scorer can't see instead of doing what you wanted. This is [Goodhart](https://arxiv.org/abs/1803.04585)'s law in miniature: once a measure becomes the target, it stops measuring what you cared about.

To anything optimizing that reward, the two outputs are indistinguishable: the same number, with no signal pulling it away from the leak. A verifiable reward score only covers what it was built to check. A high score doesn't tell you whether nothing's wrong or nobody looked. Finding out still takes a person. That's the step automation was supposed to skip.

The trade-off behind [RLVR (Reinforcement Learning from Verifiable Rewards)](https://arxiv.org/abs/2411.15124), the method Tülu 3 named and released with its full training recipe in late 2024 and that [DeepSeek-R1](https://arxiv.org/abs/2501.12948) pushed to scale two months later, is that a reward from another model's judgment can be gamed in ways that are hard to detect, while a rule-based reward is exact about what it rewards and, as you just watched, exactly as blind about everything else.

## 7. Back to your own outputs

When you ask a model to summarize a contract, draft a document, turn data into JSON, or take an action on your behalf, there's a question underneath it: *can I trust this?*

A verifier answers part of that: what's decidable from what you already know. PASS doesn't mean correct; it means the output satisfies the properties the verifier encodes, nothing more. That gap is where trust in AI systems gets misplaced, and it matters more as these systems get more to do without a person reading every output.

So trusting an AI output takes two decisions. First, is any of this decidable from what you already have? If so, build (or demand) that check; it earns you trust in that slice. Second, for whatever's left, especially the world gaps: no amount of string matching gets you there, and the honest response isn't to relax just because nothing flagged it. It's to remember what someone actually bothered to check.

## 8. [Optional] Design the coverage yourself

Everything so far, you built for one task: de-identifying a patient record. Earlier in this notebook, you saw that this isn't a one-task idea: *"ask a model to summarize a contract, draft a document, turn data into JSON... there's a question underneath: can I actually trust this?"* Here's that same question on a new task: a model turns a free-text expense description into structured JSON (a structured text format computers use to exchange data), and someone has to check it before it goes anywhere.

At the very start, this notebook also promised *"no coding background required. You'll read, predict, and press run."* Here, it bends a little: this is the one place where you write a bit of Python yourself instead of picking from options. Every step below gives you the shape you already built, ready to adapt, with a hint holding the full answer.

This is the sentence you'll be working with:

> "Lunch with client, $42.50, 2026-03-14, receipt attached."

What you're building: a verifier that takes whatever JSON a model extracts from that sentence and decides, in two steps, whether it can be trusted. The same shape as before: does the structure hold, then do the values make sense. Run the cell below.

In [ ]:
import json
from IPython.display import display, Markdown

REQUIRED_SCHEMA = {"amount": float, "category": str, "date": str, "receipt_attached": bool}
ALLOWED_CATEGORIES = {"meals", "travel", "supplies", "lodging", "other"}

EXPENSE_ATTEMPTS = {
    "attempt 1": {"amount": 42.50, "category": "meals", "receipt_attached": True},
    "attempt 2": {"amount": "42.50", "category": "meals", "date": "2026-03-14", "receipt_attached": True},
    "attempt 3": {"amount": -42.50, "category": "yacht rental", "date": "2026-03-14", "receipt_attached": True},
    "attempt 4": {"amount": 42.50, "category": "meals", "date": "2026-03-14", "receipt_attached": True},
}

print('Original sentence: "Lunch with client, $42.50, 2026-03-14, receipt attached."')
print()
print("Four things an LLM might hand back after reading the same expense report:")

blocks = []
for name, data in EXPENSE_ATTEMPTS.items():
    blocks.append("**{}**\n```json\n{}\n```".format(name, json.dumps(data, indent=2)))
display(Markdown("\n\n".join(blocks)))

### 8.1 Step 1: does the shape hold?

The question changes. Not "does a string survive," but "does every required key exist, with the right type," while the loop stays the one you already wrote.

`REQUIRED_SCHEMA`, defined in the cell above, maps each key to the type it must be:

- `str` (text)
- `bool` (`True`/`False`)
- `float` (a decimal number)

Loop over it, and for each key, check whether it's missing from `data` *or* has the wrong type; if either is true, that's a fail, same as before.

**Edit the cell below yourself:** find the line marked `YOUR CODE HERE` and replace it with that condition.

**Hint:** this is your `verify()`'s shape, from *"Your turn: build the judge"*:

    for ident in identifiers:
        if check1(ident, text):
            return Result(False, "identifier_survived", ...)
    for fact in clinical_facts:
        if check2(fact, text):
            return Result(False, "fact_lost", ...)
    return Result(True)

In [ ]:
def verify_expense_schema(data: dict) -> Result:
    for key, expected_type in REQUIRED_SCHEMA.items():
        # YOUR CODE HERE -- same shape as the loop above: is `key` missing from
        # `data`, or is `data[key]` not an instance of `expected_type`?
        if YOUR_CODE_HERE:  # replace YOUR_CODE_HERE with the condition: is `key`
                            # missing from `data`, or wrong-typed?
            return Result(False, "schema_violation", "missing or wrong-typed key: {!r}".format(key))
    return Result(True)


try:
    for name, data in EXPENSE_ATTEMPTS.items():
        print(name, ":", verify_expense_schema(data))
except Exception as e:
    print("Something broke:", repr(e))
    print("Check the line marked YOUR CODE HERE above.")

<details>
<summary style="cursor:pointer; display:inline-block; background:#e0e0e0; color:#000; padding:4px 12px; border-radius:4px; font-family:monospace; font-size:13px;">Show me the answer</summary>

    def verify_expense_schema(data: dict) -> Result:
        for key, expected_type in REQUIRED_SCHEMA.items():
            if key not in data or not isinstance(data[key], expected_type):
                return Result(False, "schema_violation", "missing or wrong-typed key: {!r}".format(key))
        return Result(True)

</details>

With your condition in place, `attempt 1` and `attempt 2` fail. One's missing a key, the other has the wrong type. `attempt 3` and `attempt 4` pass because Step 1 only checks the shape, not whether the values make sense. `attempt 3` still has a negative amount and a category nobody approved.

### 8.2 Step 2: are the values sane?

A schema-valid expense can still be nonsense: a negative amount or a category nobody approved. Write a second check. `amount` must be positive, `category` must be one of `ALLOWED_CATEGORIES`, and the check should only run after Step 1 passes, just as `verify()` always checked identifiers before facts.

**Edit the cell below yourself:** find the two lines marked `YOUR CODE HERE` and replace each with the matching condition.

**Hint:** the same "return the first failure you find" pattern as `verify()`, or `Result(True)` if nothing's wrong:

    if condition_one:
        return Result(False, ...)
    if condition_two:
        return Result(False, ...)
    return Result(True)

Your two conditions: is `data["amount"]` less than or equal to zero? Is `data["category"]` missing from `ALLOWED_CATEGORIES`?

In [ ]:
def verify_expense_values(data: dict) -> Result:
    # YOUR CODE HERE -- two checks, same "return the first failure" shape as
    # verify(): is data["amount"] <= 0? is data["category"] not in ALLOWED_CATEGORIES?
    if YOUR_CODE_HERE:  # replace with: is data["amount"] <= 0?
        return Result(False, "value_violation", "amount must be positive: {!r}".format(data["amount"]))
    if YOUR_CODE_HERE:  # replace with: is data["category"] not in ALLOWED_CATEGORIES?
        return Result(False, "value_violation", "category not allowed: {!r}".format(data["category"]))
    return Result(True)


def verify_expense(data: dict) -> Result:
    schema_result = verify_expense_schema(data)
    if not schema_result.passed:
        return schema_result
    return verify_expense_values(data)


try:
    for name, data in EXPENSE_ATTEMPTS.items():
        print(name, ":", verify_expense(data))
except Exception as e:
    print("Something broke:", repr(e))
    print("Check the two lines marked YOUR CODE HERE above.")

<details>
<summary style="cursor:pointer; display:inline-block; background:#e0e0e0; color:#000; padding:4px 12px; border-radius:4px; font-family:monospace; font-size:13px;">Show me the answer</summary>

    def verify_expense_values(data: dict) -> Result:
        if data["amount"] <= 0:
            return Result(False, "value_violation", "amount must be positive: {!r}".format(data["amount"]))
        if data["category"] not in ALLOWED_CATEGORIES:
            return Result(False, "value_violation", "category not allowed: {!r}".format(data["category"]))
        return Result(True)

</details>

Now, with both conditions in place, only `attempt 4` comes back clean. `attempt 1` and `attempt 2` still fail for the same reason as before: bad shape. `attempt 3` fails too. It has the same shape as `attempt 4`, but the amount is negative, which is what the second check was built to catch.

A missing key, a wrong type, a bad value, and one that holds up. None of that came from patient records. It came from the two moves you'd already made: check the shape, then check the values. That's the two-layer pattern: a schema check, then a value check, and it generalizes to any structured output an AI model hands you. A deterministic verifier isn't a patient-record tool that happens to work elsewhere. It's a way of thinking about any output a model hands you: what can code decide about this, and what's still yours to check by hand?

### 8.3 Your turn, on your own task

Same idea, your material. Take something an AI model actually produced for you, a summary, a rewrite, a filled-in form, and try to write down what must not survive and what must. **Edit the cell below yourself:** fill in `MY_TEXT`, `MY_IDENTIFIERS`, and `MY_FACTS`, then run it.

In [ ]:
# ---- Your turn, on your own task ------------------------------------------
# verify_v2 knows nothing about patient records. It takes a text, a list of
# strings that must not survive, and a list that must. Put in your own.

MY_TEXT        = "paste whatever a model handed you"
MY_IDENTIFIERS = []   # strings that must NOT survive
MY_FACTS       = []   # strings that must survive

print(verify_v2(MY_TEXT, MY_IDENTIFIERS, MY_FACTS))

If you can't fill in `MY_IDENTIFIERS` and `MY_FACTS` for your own task, don't treat that as a failure to complete the exercise. That is the result. Your task isn't decidable from what you currently hold. The next steps are the two you've already seen: go get the missing fact and **move it inside**, or accept that a person still has to read the output, and be specific about which part.

Same two decisions, applied to material you haven't seen before: is any of this decidable from what you already hold, and for whatever's left, who's actually reading it? The schema and JSON were never the point. They're just where the pattern happened to land this time.

## References

**Core references (the five this notebook builds on)**

*A. The mechanism at the active frontier (deterministic verification as the dominant post-training paradigm)*

- Guo, D. et al. (2025). [*DeepSeek-R1 incentivizes reasoning in LLMs through reinforcement learning.*](https://www.nature.com/articles/s41586-025-09422-z) Nature 645(8081):633-638. Preprint: [arXiv:2501.12948](https://arxiv.org/abs/2501.12948).
- Lambert, N. et al. (2024). [*Tülu 3: Pushing Frontiers in Open Language Model Post-Training.*](https://arxiv.org/abs/2411.15124) arXiv:2411.15124.
- Liu, X. et al. (2025). [*Trust, But Verify: A Self-Verification Approach to Reinforcement Learning with Verifiable Rewards.*](https://arxiv.org/abs/2505.13445) NeurIPS 2025. arXiv:2505.13445.

*B. Coverage and the verifier's limits*

- Skalse, J., Howe, N. H. R., Krasheninnikov, D. & Krueger, D. (2022). [*Defining and Characterizing Reward Gaming.*](https://proceedings.neurips.cc/paper_files/paper/2022/hash/3d719fee332caa23d5038b8a90e81796-Abstract-Conference.html) NeurIPS 2022 (preprint titled "Reward Hacking," arXiv:2209.13085).
- Xin, H. et al. (2025). [*DeepSeek-Prover-V1.5: Harnessing Proof Assistant Feedback for Reinforcement Learning and Monte-Carlo Tree Search.*](https://arxiv.org/abs/2408.08152) ICLR 2025. arXiv:2408.08152.

**Verification foundations**

- Turing, A. M. (1936). [*On Computable Numbers, with an Application to the Entscheidungsproblem.*](https://doi.org/10.1112/plms/s2-42.1.230) Proc. London Math. Soc. s2-42(1):230-265.
- Hoare, C. A. R. (1969). [*An Axiomatic Basis for Computer Programming.*](https://doi.org/10.1145/363235.363259) CACM 12(10):576-580.
- Sipser, M. (2013). [*Introduction to the Theory of Computation*](https://www.cengage.com/c/introduction-to-the-theory-of-computation-3e-sipser/9781133187790/), 3rd ed., Cengage. Ch. 4, "Decidability."
- Cobbe, K. et al. (2021). [*Training Verifiers to Solve Math Word Problems.*](https://arxiv.org/abs/2110.14168) arXiv:2110.14168.
- [ISO 13616-1:2020](https://www.iso.org/standard/81090.html). *Financial services. International bank account number (IBAN). Part 1: Structure of the IBAN.*
- [ISO/IEC 7064:2003](https://www.iso.org/standard/31531.html). *Information technology. Security techniques. Check character systems* (defines the MOD 97-10 system used by IBAN).

**LLM-as-judge and self-assessment**

- Zheng, L. et al. (2023). [*Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.*](https://arxiv.org/abs/2306.05685) NeurIPS 2023, Datasets & Benchmarks Track. arXiv:2306.05685.
- Huang, J. et al. (2024). [*Large Language Models Cannot Self-Correct Reasoning Yet.*](https://arxiv.org/abs/2310.01798) ICLR 2024. arXiv:2310.01798.

**Reward and training**

- Yue, Y. et al. (2025). [*Does Reinforcement Learning Really Incentivize Reasoning Capacity in LLMs Beyond the Base Model?*](https://arxiv.org/abs/2504.13837) NeurIPS 2025 (Oral). arXiv:2504.13837.
- Krakovna, V. et al. (2020). [*Specification gaming: the flip side of AI ingenuity.*](https://deepmind.google/blog/specification-gaming-the-flip-side-of-ai-ingenuity/) DeepMind blog.
- Goodhart, C. A. E. (1975). *Problems of Monetary Management: The U.K. Experience.* Papers in Monetary Economics, Vol. I, Reserve Bank of Australia; [reprinted 1984](https://link.springer.com/chapter/10.1007/978-1-349-17295-5_4), Macmillan.
- Manheim, D. & Garrabrant, S. (2018). [*Categorizing Variants of Goodhart's Law.*](https://arxiv.org/abs/1803.04585) arXiv:1803.04585.
- Sutton, R. S. & Barto, A. G. (2018). [*Reinforcement Learning: An Introduction*](http://incompleteideas.net/book/the-book-2nd.html), 2nd ed., MIT Press.

**Repair loop, rejection sampling, self-correction**

- Bishop, C. M. (2006). [*Pattern Recognition and Machine Learning*](https://www.microsoft.com/en-us/research/uploads/prod/2006/01/Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf), Springer. §11.1.2, "Rejection sampling."
- Touvron, H. et al. (2023). [*Llama 2: Open Foundation and Fine-Tuned Chat Models.*](https://arxiv.org/abs/2307.09288) arXiv:2307.09288 (rejection sampling in LLM post-training).
- Chen, X. et al. (2024). [*Teaching Large Language Models to Self-Debug.*](https://arxiv.org/abs/2304.05128) ICLR 2024. arXiv:2304.05128.
- Olausson, T. X. et al. (2024). [*Is Self-Repair a Silver Bullet for Code Generation?*](https://arxiv.org/abs/2306.09896) ICLR 2024. arXiv:2306.09896.
- Pan, L. et al. (2024). [*Automatically Correcting Large Language Models: Surveying the Landscape of Diverse Automated Correction Strategies.*](https://aclanthology.org/2024.tacl-1.27/) TACL 12:484-506.

**Medical privacy and de-identification**

- HIPAA Safe Harbor, [45 CFR § 164.514(b)(2)](https://www.ecfr.gov/current/title-45/subtitle-A/subchapter-C/part-164/subpart-E/section-164.514) (eighteen identifier categories); [HHS de-identification guidance](https://www.hhs.gov/hipaa/for-professionals/special-topics/de-identification/index.html).
- Garfinkel, S. (2015). [*De-Identification of Personal Information.*](https://csrc.nist.gov/pubs/ir/8053/final) NISTIR 8053.
- Garfinkel, S., Guttman, B., Near, J., Dajani, A. & Singer, P. (2023). [*De-Identifying Government Datasets: Techniques and Governance.*](https://csrc.nist.gov/pubs/sp/800/188/final) NIST SP 800-188.
- Meystre, S. M., Friedlin, F. J., South, B. R., Shen, S. & Samore, M. H. (2010). [*Automatic de-identification of textual documents in the electronic health record: a review of recent research.*](https://doi.org/10.1186/1471-2288-10-70) BMC Med. Res. Methodol. 10:70.
- Sweeney, L. (2002). [*k-Anonymity: A Model for Protecting Privacy.*](https://doi.org/10.1142/S0218488502001648) Int. J. Uncertain. Fuzziness Knowl.-Based Syst. 10(5):557-570.

**Errors, gaps, and extensions**

- Fawcett, T. (2006). [*An Introduction to ROC Analysis.*](https://doi.org/10.1016/j.patrec.2005.10.010) Pattern Recognition Letters 27(8):861-874.
- Green, D. M. & Swets, J. A. (1966). *Signal Detection Theory and Psychophysics.* Wiley.
- Aho, A. V. & Corasick, M. J. (1975). [*Efficient String Matching: An Aid to Bibliographic Search.*](https://doi.org/10.1145/360825.360855) CACM 18(6):333-340.
- [Unicode Standard Annex #29 (UAX #29)](https://www.unicode.org/reports/tr29/). *Unicode Text Segmentation.* Unicode 17.0.0.
- Shen, W., Wang, J. & Han, J. (2015). [*Entity Linking with a Knowledge Base: Issues, Techniques, and Solutions.*](https://doi.org/10.1109/TKDE.2014.2327028) IEEE TKDE 27(2):443-460.
- Bunescu, R. & Pașca, M. (2006). [*Using Encyclopedic Knowledge for Named Entity Disambiguation.*](https://aclanthology.org/E06-1002/) EACL 2006, 9-16.
- Lewis, P. et al. (2020). [*Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.*](https://arxiv.org/abs/2005.11401) NeurIPS 2020. arXiv:2005.11401.
- de Moura, L. & Ullrich, S. (2021). [*The Lean 4 Theorem Prover and Programming Language.*](https://doi.org/10.1007/978-3-030-79876-5_37) CADE-28.

Terminology introduced here, not drawn from the literature: **loud** and **quiet failures** (false-positive and false-negative cells), **string gap** and **world gap**, and the **two-layer verification pattern**.